In [2]:
%pip install bs4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import requests
import bs4 as bs
import urllib.request
from io import StringIO

## Extracting features of 2020 movies from Wikipedia

In [4]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2020"

In [9]:
url = "https://en.wikipedia.org/wiki/List_of_American_films_of_2020"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

tables = pd.read_html(StringIO(response.text))

df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]

In [11]:
source = urllib.request.urlopen(link).read()
soup = bs.BeautifulSoup(source,'lxml')

HTTPError: HTTP Error 403: Forbidden

In [10]:
tables = soup.find_all('table',class_='wikitable sortable')

NameError: name 'soup' is not defined

In [5]:
len(tables)

4

In [6]:
type(tables[0])

bs4.element.Tag

In [7]:
df1 = pd.read_html(str(tables[0]))[0]
df2 = pd.read_html(str(tables[1]))[0]
df3 = pd.read_html(str(tables[2]))[0]
df4 = pd.read_html(str(tables[3]).replace("'1\"\'",'"1"'))[0] # avoided "ValueError: invalid literal for int() with base 10: '1"'

In [8]:
df = df1.append(df2.append(df3.append(df4,ignore_index=True),ignore_index=True),ignore_index=True)

In [12]:
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

In [13]:
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,3,The Grudge,Screen Gems / Stage 6 Films / Ghost House Pict...,Nicolas Pesce (director/screenplay); Andrea Ri...,[2]
1,J A N U A R Y,10,Underwater,20th Century Fox / Chernin Entertainment,"William Eubank (director); Brian Duffield, Ada...",[3]
2,J A N U A R Y,10,Like a Boss,Paramount Pictures / Artists First,"Miguel Arteta (director); Sam Pitman, Adam Col...",[4]
3,J A N U A R Y,10,Three Christs,IFC Films,Jon Avnet (director/screenplay); Eric Nazarian...,NaN
4,J A N U A R Y,10,Inherit the Viper,Lionsgate / Barry Films / Tycor International ...,Anthony Jerjen (director); Andrew Crabtree (sc...,[5]
...,...,...,...,...,...,...
274,D E C E M B E R,25,We Can Be Heroes,Netflix / Troublemaker Studios / Double R Prod...,Robert Rodriguez (director/screenplay); Priyan...,[247]
275,D E C E M B E R,25,News of the World,Universal Pictures / Playtone / Perfect World ...,Paul Greengrass (director/screenplay); Luke Da...,[248]
276,D E C E M B E R,25,One Night in Miami...,Amazon Studios,Regina King (director); Kemp Powers (screenpla...,[249]
277,D E C E M B E R,25,Promising Young Woman,Focus Features / FilmNation Entertainment,Emerald Fennell (director/screenplay); Carey M...,[250]


In [14]:
df_2020 = df[['Title','Cast and crew']]

In [15]:
df_2020

,Title,Cast and crew
0,The Grudge,Nicolas Pesce (director/screenplay); Andrea Ri...
1,Underwater,"William Eubank (director); Brian Duffield, Ada..."
2,Like a Boss,"Miguel Arteta (director); Sam Pitman, Adam Col..."
3,Three Christs,Jon Avnet (director/screenplay); Eric Nazarian...
4,Inherit the Viper,Anthony Jerjen (director); Andrew Crabtree (sc...
...,...,...
274,We Can Be Heroes,Robert Rodriguez (director/screenplay); Priyan...
275,News of the World,Paul Greengrass (director/screenplay); Luke Da...
276,One Night in Miami...,Regina King (director); Kemp Powers (screenpla...
277,Promising Young Woman,Emerald Fennell (director/screenplay); Carey M...


In [17]:
%pip install tmdbv3api

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from tmdbv3api import TMDb
import json
import requests
tmdb = TMDb()
tmdb.api_key = 'da8a88c9410d08718361982f107d161f'

In [19]:
from tmdbv3api import Movie
tmdb_movie = Movie() 
def get_genre(x):
    genres = []
    result = tmdb_movie.search(x)
    if not result:
      return np.nan
    else:
      movie_id = result[0].id
      response = requests.get('https://api.themoviedb.org/3/movie/{}?api_key={}'.format(movie_id,tmdb.api_key))
      data_json = response.json()
      if data_json['genres']:
          genre_str = " " 
          for i in range(0,len(data_json['genres'])):
              genres.append(data_json['genres'][i]['name'])
          return genre_str.join(genres)
      else:
          return np.nan

In [24]:
df1 = df[['Title']]

print(df1.head())

               Title
0         The Grudge
1         Underwater
2        Like a Boss
3      Three Christs
4  Inherit the Viper


In [21]:
import requests
import numpy as np
import time

API_KEY = "da8a88c9410d08718361982f107d161f"

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Connection": "keep-alive"
})

def get_movie_genres(title):

    for attempt in range(3):

        try:

            # SEARCH MOVIE
            search_url = "https://api.themoviedb.org/3/search/movie"

            search_response = session.get(
                search_url,
                params={
                    "api_key": API_KEY,
                    "query": title
                },
                timeout=30
            )

            search_data = search_response.json()

            if len(search_data["results"]) == 0:
                return np.nan

            movie_id = search_data["results"][0]["id"]

            # MOVIE DETAILS
            details_url = f"https://api.themoviedb.org/3/movie/{movie_id}"

            details_response = session.get(
                details_url,
                params={
                    "api_key": API_KEY
                },
                timeout=30
            )

            details_data = details_response.json()

            genres = [g["name"] for g in details_data["genres"]]

            time.sleep(0)

            return " ".join(genres)

        except Exception as e:

            print(f"Retry {attempt+1} failed for {title}")

            time.sleep(5)

    return np.nan

In [25]:
df1 = df1.copy()

genres_list = []

for title in df1["Title"]:

    genres = get_movie_genres(title)

    print(title, "->", genres)

    genres_list.append(genres)

df["genres"] = genres_list

print(df)

The Grudge -> Horror Mystery
Underwater -> Horror Science Fiction Action Adventure
Like a Boss -> Comedy
Three Christs -> Drama
Inherit the Viper -> Crime Thriller Drama
The Sonata -> Horror Thriller Mystery
The Murder of Nicole Brown Simpson -> Documentary
Angels Fallen -> Action Romance Crime
Bad Boys for Life -> Thriller Action Crime
Dolittle -> Family Comedy Fantasy Adventure
A Fall from Grace -> Drama Thriller Mystery
The Gentlemen -> Action Comedy Crime
The Turning -> Horror Thriller
The Last Full Measure -> Drama Action War
John Henry -> Family
The Rhythm Section -> Action Thriller
Gretel & Hansel -> Fantasy Horror Mystery
The Assistant -> Horror Mystery
Birds of Prey -> Comedy Horror Thriller
The Lodge -> Horror Drama Mystery Thriller
Timmy Failure: Mistakes Were Made -> Family Comedy Fantasy Mystery
Horse Girl -> Drama
To All the Boys: P.S. I Still Love You -> Romance Comedy
Sonic the Hedgehog -> Action Science Fiction Comedy Family
Fantasy Island -> Adventure Fantasy Horror M

In [26]:
df_2020 = df[['Title','Cast and crew','genres']]

In [27]:
df_2020

,Title,Cast and crew,genres
0,The Grudge,Nicolas Pesce (director/screenplay); Andrea Ri...,Horror Mystery
1,Underwater,"William Eubank (director); Brian Duffield, Ada...",Horror Science Fiction Action Adventure
2,Like a Boss,"Miguel Arteta (director); Sam Pitman, Adam Col...",Comedy
3,Three Christs,Jon Avnet (director/screenplay); Eric Nazarian...,Drama
4,Inherit the Viper,Anthony Jerjen (director); Andrew Crabtree (sc...,Crime Thriller Drama
...,...,...,...
274,We Can Be Heroes,Robert Rodriguez (director/screenplay); Priyan...,Family Action Fantasy Comedy
275,News of the World,Paul Greengrass (director/screenplay); Luke Da...,Drama Western Adventure
276,One Night in Miami...,Regina King (director); Kemp Powers (screenpla...,Drama
277,Promising Young Woman,Emerald Fennell (director/screenplay); Carey M...,Thriller Crime Drama


In [28]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x: 
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [29]:
df_2020['director_name'] = df_2020['Cast and crew'].map(lambda x: get_director(str(x)))

C:\Users\Admin\AppData\Local\Temp\ipykernel_4808\3848435694.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2020['director_name'] = df_2020['Cast and crew'].map(lambda x: get_director(str(x)))


In [30]:
def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(", ")[0])

In [31]:
df_2020['actor_1_name'] = df_2020['Cast and crew'].map(lambda x: get_actor1(str(x)))

C:\Users\Admin\AppData\Local\Temp\ipykernel_4808\3646981363.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2020['actor_1_name'] = df_2020['Cast and crew'].map(lambda x: get_actor1(str(x)))


In [34]:
def get_actor2(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 2:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[1])

In [35]:
df_2020['actor_2_name'] = df_2020['Cast and crew'].map(lambda x: get_actor2(str(x)))

C:\Users\Admin\AppData\Local\Temp\ipykernel_4808\463054939.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2020['actor_2_name'] = df_2020['Cast and crew'].map(lambda x: get_actor2(str(x)))


In [36]:
def get_actor3(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 3:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[2])

In [37]:
df_2020['actor_3_name'] = df_2020['Cast and crew'].map(lambda x: get_actor3(str(x)))

In [25]:
df_2020

,Title,Cast and crew,genres,director_name,actor_1_name,actor_2_name,actor_3_name
0,The Grudge,Nicolas Pesce (director/screenplay); Andrea Ri...,Horror Mystery Thriller,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho
1,Underwater,"William Eubank (director); Brian Duffield, Ada...",Action Horror Science Fiction Thriller,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick
2,Three Christs,Jon Avnet (director/screenplay); Eric Nazarian...,Drama,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins
3,Like a Boss,"Miguel Arteta (director); Sam Pitman, Adam Col...",Comedy,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek
4,Inherit the Viper,Anthony Jerjen (director); Andrew Crabtree (sc...,Drama Thriller Crime,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs
...,...,...,...,...,...,...,...
250,News of the World,Paul Greengrass (director/screenplay); Luke Da...,Drama Western,Paul Greengrass,Tom Hanks,Helena Zengel,NaN
251,One Night in Miami,Regina King (director); Kemp Powers (screenpla...,Drama,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge
252,Promising Young Woman,Emerald Fennell (director/screenplay); Carey M...,Thriller Crime Drama,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie
253,Sylvie's Love,Eugene Ashe (director/screenplay); Tessa Thomp...,Drama,Eugene Ashe,Tessa Thompson,Nnamdi Asomugha,Ryan Michelle Bathe


In [38]:
df_2020 = df_2020.rename(columns={'Title':'movie_title'})

In [39]:
new_df20 = df_2020.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [40]:
new_df20

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho,Horror Mystery,The Grudge
1,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick,Horror Science Fiction Action Adventure,Underwater
2,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek,Comedy,Like a Boss
3,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins,Drama,Three Christs
4,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs,Crime Thriller Drama,Inherit the Viper
...,...,...,...,...,...,...
274,Robert Rodriguez,Priyanka Chopra Jonas,Pedro Pascal,YaYa Gosselin,Family Action Fantasy Comedy,We Can Be Heroes
275,Paul Greengrass,Tom Hanks,Helena Zengel,NaN,Drama Western Adventure,News of the World
276,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge,Drama,One Night in Miami...
277,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie,Thriller Crime Drama,Promising Young Woman


In [41]:
new_df20['comb'] = new_df20['actor_1_name'] + ' ' + new_df20['actor_2_name'] + ' '+ new_df20['actor_3_name'] + ' '+ new_df20['director_name'] +' ' + new_df20['genres']

In [42]:
new_df20.isna().sum()

director_name     0
actor_1_name      0
actor_2_name      5
actor_3_name     29
genres            1
movie_title       0
comb             29
dtype: int64

In [43]:
new_df20 = new_df20.dropna(how='any')

In [44]:
new_df20.isna().sum()

director_name    0
actor_1_name     0
actor_2_name     0
actor_3_name     0
genres           0
movie_title      0
comb             0
dtype: int64

In [45]:
new_df20['movie_title'] = new_df20['movie_title'].str.lower()

C:\Users\Admin\AppData\Local\Temp\ipykernel_4808\2267385682.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df20['movie_title'] = new_df20['movie_title'].str.lower()


In [46]:
new_df20

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho,Horror Mystery,the grudge,Andrea Riseborough Demián Bichir John Cho Nico...
1,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick,Horror Science Fiction Action Adventure,underwater,Kristen Stewart Vincent Cassel Jessica Henwick...
2,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek,Comedy,like a boss,Tiffany Haddish Rose Byrne Salma Hayek Miguel ...
3,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins,Drama,three christs,Richard Gere Peter Dinklage Walton Goggins Jon...
4,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs,Crime Thriller Drama,inherit the viper,Josh Hartnett Margarita Levieva Chandler Riggs...
...,...,...,...,...,...,...,...
273,Pete Docter,Jamie Foxx,Tina Fey,Graham Norton,Animation Family Drama Music Fantasy,soul,Jamie Foxx Tina Fey Graham Norton Pete Docter ...
274,Robert Rodriguez,Priyanka Chopra Jonas,Pedro Pascal,YaYa Gosselin,Family Action Fantasy Comedy,we can be heroes,Priyanka Chopra Jonas Pedro Pascal YaYa Gossel...
276,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge,Drama,one night in miami...,Kingsley Ben-Adir Eli Goree Aldis Hodge Regina...
277,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie,Thriller Crime Drama,promising young woman,Carey Mulligan Bo Burnham Alison Brie Emerald ...


In [47]:
old_df = pd.read_csv('final_data.csv')

In [48]:
old_df

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Joachim Rønning Espen Sandberg,Johnny Depp,Javier Bardem,Geoffrey Rush,Adventure Action Fantasy Comedy,pirates of the caribbean: dead men tell no tales,Johnny Depp Javier Bardem Geoffrey Rush Joachi...
1,Zack Snyder,Ben Affleck,Henry Cavill,Gal Gadot,Action Adventure Fantasy Science Fiction,justice league,Ben Affleck Henry Cavill Gal Gadot Zack Snyder...
2,Taika Waititi,Chris Hemsworth,Tom Hiddleston,Cate Blanchett,Action Adventure Fantasy Science Fiction,thor: ragnarok,Chris Hemsworth Tom Hiddleston Cate Blanchett ...
3,James Gunn,Chris Pratt,Zoe Saldana,Dave Bautista,Action Adventure Comedy Science Fiction,guardians of the galaxy vol. 2,Chris Pratt Zoe Saldana Dave Bautista James Gu...
4,Sean McNamara,Pierce Brosnan,William Hurt,Benjamin Walker,Fantasy Action Adventure,the king's daughter,Pierce Brosnan William Hurt Benjamin Walker Se...
...,...,...,...,...,...,...,...
956,"Nick Bruno, Troy Quane",Nick Bruno,Tom Holland,Rashida Jones,Animation Action Adventure Comedy Family,spies in disguise,Nick Bruno Tom Holland Rashida Jones Nick Brun...
957,Greta Gerwig,Greta Gerwig (director/screenplay); Saoirse Ronan,Emma Watson,Florence Pugh,Drama Romance,little women,Greta Gerwig (director/screenplay); Saoirse Ro...
958,Sam Mendes,Sam Mendes (director/screenplay); Krysty Wilso...,Dean-Charles Chapman,Mark Strong,War History Drama,1917,Sam Mendes (director/screenplay); Krysty Wilso...
959,Destin Daniel Cretton,Destin Daniel Cretton (director/screenplay),Jamie Foxx,Brie Larson,Drama Crime History,just mercy,Destin Daniel Cretton (director/screenplay) Ja...


In [49]:
final_df = pd.concat([old_df, new_df20], ignore_index=True)

In [50]:
final_df

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Joachim Rønning Espen Sandberg,Johnny Depp,Javier Bardem,Geoffrey Rush,Adventure Action Fantasy Comedy,pirates of the caribbean: dead men tell no tales,Johnny Depp Javier Bardem Geoffrey Rush Joachi...
1,Zack Snyder,Ben Affleck,Henry Cavill,Gal Gadot,Action Adventure Fantasy Science Fiction,justice league,Ben Affleck Henry Cavill Gal Gadot Zack Snyder...
2,Taika Waititi,Chris Hemsworth,Tom Hiddleston,Cate Blanchett,Action Adventure Fantasy Science Fiction,thor: ragnarok,Chris Hemsworth Tom Hiddleston Cate Blanchett ...
3,James Gunn,Chris Pratt,Zoe Saldana,Dave Bautista,Action Adventure Comedy Science Fiction,guardians of the galaxy vol. 2,Chris Pratt Zoe Saldana Dave Bautista James Gu...
4,Sean McNamara,Pierce Brosnan,William Hurt,Benjamin Walker,Fantasy Action Adventure,the king's daughter,Pierce Brosnan William Hurt Benjamin Walker Se...
...,...,...,...,...,...,...,...
1206,Pete Docter,Jamie Foxx,Tina Fey,Graham Norton,Animation Family Drama Music Fantasy,soul,Jamie Foxx Tina Fey Graham Norton Pete Docter ...
1207,Robert Rodriguez,Priyanka Chopra Jonas,Pedro Pascal,YaYa Gosselin,Family Action Fantasy Comedy,we can be heroes,Priyanka Chopra Jonas Pedro Pascal YaYa Gossel...
1208,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge,Drama,one night in miami...,Kingsley Ben-Adir Eli Goree Aldis Hodge Regina...
1209,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie,Thriller Crime Drama,promising young woman,Carey Mulligan Bo Burnham Alison Brie Emerald ...


In [51]:
final_df.to_csv('main_data.csv',index=False)